# 日本語の共参照解析をする (KWJA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_semantics/coref_kwja.ipynb)

NLP クックブック 19-03 ／ 意味解析

「彼」「その会社」「同社」が、文章中のどの語と同じものを指しているのかを知りたい。

In [1]:
!pip install -q kwja

In [2]:
import subprocess
from rhoknp import Document

def kwja(text, model_size="base"):
    result = subprocess.run(["kwja", "--text", text, "--tasks", "char,word", "--model-size", model_size],
                            capture_output=True, text=True, check=True)
    return Document.from_knp(result.stdout)

text = "山田社長は新製品を発表した。彼はこの製品が同社の主力になると述べた。山田氏によると、発売は来年の春だという。"
doc = kwja(text)

# 同じものを指す基本句をまとめる (共参照クラスタ)
clusters, seen = [], set()
for bp in doc.base_phrases:
    if bp.global_index in seen:
        continue
    group = [bp] + list(bp.get_coreferents())
    if len(group) > 1:
        seen.update(g.global_index for g in group)
        clusters.append(sorted(group, key=lambda g: g.global_index))
for i, group in enumerate(clusters):
    print(i, [(g.text, f"{g.sentence.sid.split('-')[-1]} 文目") for g in group])

0 [('山田', '1 文目'), ('社長は', '1 文目'), ('彼は', '2 文目'), ('山田氏に', '3 文目')]
1 [('製品を', '1 文目'), ('製品が', '2 文目')]


In [3]:
# 固有表現と、各基本句の意味的な素性もまとめて得られる
print([(ne.text, ne.category.value) for ne in doc.named_entities])
for bp in doc.sentences[0].base_phrases:
    print(bp.text, [k for k in bp.features if k in ("体言", "用言", "SM-主体", "時制", "節-主辞")], bp.features.get("時制", ""))

[('山田', 'PERSON'), ('山田', 'PERSON'), ('来年', 'DATE'), ('春', 'DATE')]
山田 ['体言', 'SM-主体'] 
社長は ['体言', 'SM-主体'] 
新 [] 
製品を ['体言'] 
発表した。 ['用言', '時制', '節-主辞'] 過去


- 日本語は代名詞を使わずに省略することが多い。省略された項の補完は、前項の述語項構造解析（ゼロ照応）で扱われる。共参照解析は、書かれている語同士の同一性を扱う。
- 「同社」のように、指示先が文中にはっきり書かれていない場合（ここでは「山田社長」の会社）は解決できない。橋渡し照応と呼ばれる、より難しい問題。
- 精度は文体に依存する。ニュースや Wikipedia のような整った文に強く、会話や SNS では落ちる。